In [7]:
# ---------- CELL 0: locate the project (same in every notebook) ----------
import os, sys
from pathlib import Path

def find_root(marker="PROJECT_ROOT.marker"):
    here = Path.cwd().resolve()                      # start where this notebook runs
    for folder in [here, *here.parents]:             # go UP one level at a time
        if (folder / marker).exists():
            return folder
    if "google.colab" in sys.modules:                # Colab: look inside Google Drive
        from google.colab import drive
        drive.mount("/content/drive")
        for pattern in (marker, f"*/{marker}", f"*/*/{marker}", f"*/*/*/{marker}"):
            hits = sorted(Path("/content/drive/MyDrive").glob(pattern))
            if hits:
                return hits[0].parent
    raise FileNotFoundError("PROJECT_ROOT.marker not found - finish Day 1 first.")

ROOT = find_root()
os.chdir(ROOT / "notebooks")
RAW, CLEAN = ROOT/"data"/"raw", ROOT/"data"/"clean"
FEATURES, MODEL_OUT = ROOT/"data"/"features", ROOT/"data"/"model_outputs"
SQL_EXPORTS, BI_READY = ROOT/"data"/"sql_exports", ROOT/"data"/"bi_ready"
MODELS, REPORTS, DOCS = ROOT/"models", ROOT/"reports", ROOT/"docs"
FIGURES = REPORTS/"figures"
sys.path.append(str(ROOT/"src"))
print("Project root:", ROOT)

Project root: /content/drive/MyDrive/Internmo/netflix_churn_project


In [8]:
# ---------- CELL 1: Power BI theme ----------
import json
import numpy as np
import pandas as pd

theme = {
    "name": "Netflix Churn",
    "dataColors": ["#B20710", "#221F1F", "#E50914", "#831010", "#564D4D", "#B3B3B3", "#E87C03", "#5E5E5E"],
    "background": "#FFFFFF",
    "foreground": "#221F1F",
    "tableAccent": "#B20710",
    "textClasses": {
        "title":   {"fontSize": 14, "fontFace": "Segoe UI Semibold", "color": "#221F1F"},
        "label":   {"fontSize": 10, "fontFace": "Segoe UI", "color": "#564D4D"},
        "callout": {"fontSize": 28, "fontFace": "Segoe UI Semibold", "color": "#B20710"},
    },
}
(ROOT / "powerbi" / "netflix_theme.json").write_text(json.dumps(theme, indent=2))
print("theme written: powerbi/netflix_theme.json")

theme written: powerbi/netflix_theme.json


In [9]:
# ---------- CELL 2: load everything the report needs ----------
kpi       = pd.read_csv(BI_READY / "kpi_summary.csv").iloc[0]
eda       = pd.read_csv(REPORTS / "eda_churn_by_group.csv")
themes    = pd.read_csv(MODEL_OUT / "text_theme_summary.csv")
segments  = pd.read_csv(MODEL_OUT / "segment_profile.csv")
scenarios = pd.read_csv(MODEL_OUT / "campaign_scenarios.csv")
countries = pd.read_csv(SQL_EXPORTS / "q30_country_churn_summary.csv")
test_pred = pd.read_csv(MODEL_OUT / "churn_test_predictions.csv")
cleaning  = pd.read_csv(DOCS / "cleaning_log.csv")
value     = pd.read_csv(MODEL_OUT / "customer_value.csv")

# churn rate in the model's riskiest 20% (test set)
cut = test_pred["churn_probability"].quantile(0.80)
top20_rate = 100 * test_pred.loc[test_pred["churn_probability"] >= cut, "churned"].mean()
overall_test_rate = 100 * test_pred["churned"].mean()

# the groups whose churn differs most from the overall rate (at least 100 customers)
eligible = eda[eda["customers"] >= 100].copy()
eligible["abs_gap"] = eligible["gap_vs_overall_pts"].abs()
top_gaps = eligible.sort_values("abs_gap", ascending=False).head(5)
gap_lines = "\n".join(
    f"- **{r.category} = {r.group}**: {r.churn_rate_pct:.1f}% churn ({r.gap_vs_overall_pts:+.1f} points vs overall, {int(r.customers):,} customers)"
    for r in top_gaps.itertuples())

feedback_themes = themes[themes["source"] == "feedback"].sort_values("churn_rate_pct", ascending=False)
worst_theme = feedback_themes.iloc[0]
best_segment_risk = segments.sort_values("expected_revenue_at_risk", ascending=False).iloc[0]

pivot = scenarios.pivot(index="campaign_size", columns="targeting", values="net_value")
pivot["extra"] = pivot["model-targeted"] - pivot["random"]
best_campaign = pivot["extra"].idxmax()
top3_countries = ", ".join(f"{r.country} ({r.churn_rate_pct:.1f}%)" for r in countries.head(3).itertuples())
print("numbers ready")

numbers ready


In [10]:
# ---------- CELL 3: the report ----------
report = f"""# Netflix Customer Churn Intelligence: Final Project Report

*Draft generated from the project files. Read it, check each claim against the dashboard, and rewrite the
summary and recommendations in your own words before sharing.*

## 1. Executive summary
- **{kpi['churn_rate_pct']:.1f}%** of the {int(kpi['total_customers']):,} customers have churned ({int(kpi['churned_customers']):,}); {int(kpi['active_customers']):,} are still active, paying **{kpi['monthly_recurring_revenue']:,.0f}** per month.
- A churn model ranks active customers by risk. It is **modest but real**: test AUC {kpi['churn_model_test_auc']:.3f}; the riskiest 20% churn at **{top20_rate:.1f}%** versus {overall_test_rate:.1f}% on average.
- {int(kpi['high_risk_customers']):,} active customers are in the High-risk tier. Estimated 12-month revenue at risk across all active customers is **{kpi['value_at_risk_12m']:,.0f}**.
- Customers who commented about **{worst_theme['theme_name']}** churn at **{worst_theme['churn_rate_pct']:.1f}%**, the highest of any feedback theme.
- The segment with the most expected revenue at risk is **{best_segment_risk['segment_name']}**.

## 2. Business question
Which customers are likely to leave, why, how much revenue is at stake, and where should a retention team act first?

## 3. Data
Nine tables for 8,000 customers (customers, subscriptions, plans, payments, viewing activity, content, support tickets, feedback, churn labels).
The data is synthetic. Analysis date (as-of) is **2026-06-30**. Raw files were never edited; {len(cleaning)} cleaning actions are listed in `docs/cleaning_log.csv`.

## 4. Method (what was done, in order)
1. **Audit and clean (Python):** duplicates removed, country and device names standardised, impossible ages and outliers handled, payment `subscription_id` recovered by date matching, odd payment amounts corrected (originals kept).
2. **SQL (MySQL):** database built from the clean files; all 37 business questions answered (views for the current plan, risk signals and monthly viewing).
3. **Exploratory analysis (Python):** 33 charts; Python and SQL results cross-checked.
4. **Features:** built at a snapshot date (the day before the first recorded churn) so no feature uses future information; a leakage test proved it.
5. **Churn model:** three models compared by cross-validation, tuned, and evaluated once on a locked test set.
6. **Segmentation:** K-Means with 5 behavioural segments.
7. **Customer value:** expected 12-month value from plan price and churn probability.
8. **Text themes:** keyword rules for feedback, TF-IDF clustering for tickets (free tools only).
9. **Dashboard:** Power BI reads only `data/bi_ready`; SQL and Python results are displayed, not recalculated.

## 5. Key findings
**Where churn differs most from the overall rate ({kpi['churn_rate_pct']:.1f}%):**
{gap_lines}

**Highest-churn countries (SQL Q30):** {top3_countries}.

**Model.** AUC {kpi['churn_model_test_auc']:.3f}. A leaky version scored higher (see `reports/churn_model_metrics.csv`), which is why feature timing mattered more than the choice of algorithm.

**Segments.** {best_segment_risk['segment_name']} carries the most expected monthly revenue at risk ({best_segment_risk['expected_revenue_at_risk']:,.0f}). See `data/model_outputs/segment_profile.csv`.

**Retention campaign (illustrative assumptions).** Model-targeting beats random targeting most for the **{best_campaign}** campaign (extra net value {pivot.loc[best_campaign, 'extra']:,.0f}).

**YOUR OWN FINDINGS:** (add 3 sentences from `docs/eda_findings.md` in your own words)

## 6. Recommendations
1. Start retention outreach with the High-risk tier, sorted by value at risk (`customer_value.csv`, dashboard page 4).
2. Fix app stability and payment failures first. Customers who complain about the app churn most, and failed payments are the strongest numeric signal in the model.
3. Run a **controlled test** (contact half of the High tier, hold out the rest) to measure the real save rate before scaling.
4. Replace the example campaign costs and save rate with real figures.
5. Re-train as more months of churn data arrive.

## 7. Limitations (be upfront about these)
- Synthetic data with weak signals; AUC around 0.57 is the honest ceiling here.
- Churn dates exist only from 2025-10-03, so there is one training snapshot and no out-of-time test.
- Viewing data covers 12 months; payments stop on 2026-05-31 and hold at most 15 payments per customer. For this reason no lifetime-value regression was built.
- `auto_renew` is stored as extracted, not as of the snapshot.
- Text is templated (13 feedback comments, 32 ticket descriptions); theme results are cleaner than real text would be.
- Theme and segment churn rates are descriptive, not causal.

## 8. Reproducibility
Run order and file locations are in `README.md`. All paths are found automatically from the project folder; the only typed paths are the folder placeholder
in `database/02_load_data.sql` (replaced once) and the `DataFolder` parameter in Power BI.
"""
(REPORTS / "Final_Project_Report.md").write_text(report)
print(report[:1800])

# Netflix Customer Churn Intelligence: Final Project Report

*Draft generated from the project files. Read it, check each claim against the dashboard, and rewrite the
summary and recommendations in your own words before sharing.*

## 1. Executive summary
- **24.9%** of the 8,000 customers have churned (1,991); 6,009 are still active, paying **81,365** per month.
- A churn model ranks active customers by risk. It is **modest but real**: test AUC 0.568; the riskiest 20% churn at **31.2%** versus 24.5% on average.
- 1,201 active customers are in the High-risk tier. Estimated 12-month revenue at risk across all active customers is **151,335**.
- Customers who commented about **App stability** churn at **33.2%**, the highest of any feedback theme.
- The segment with the most expected revenue at risk is **High-Value Loyalists**.

## 2. Business question
Which customers are likely to leave, why, how much revenue is at stake, and where should a retention team act first?

## 3. Data
Nine tables

In [11]:
# ---------- CELL 4: presentation notes ----------
notes = f"""# Presentation Notes

## 60-second talk track
"I built an end-to-end churn analysis on 8,000 Netflix-style customers. I cleaned nine tables in Python, loaded them into
MySQL and answered 37 business questions, then built a churn model and a segmentation, and put everything in a Power BI
dashboard. About {kpi['churn_rate_pct']:.0f}% of customers have churned. The model is modest (AUC {kpi['churn_model_test_auc']:.2f}), but it
finds {top20_rate:.0f}% churners in the riskiest 20% against {overall_test_rate:.0f}% on average, and it shows where revenue at risk is concentrated.
I was careful to prevent data leakage and I am upfront about the limits of the data."

## Questions you should expect, and honest answers
**Why is the AUC only about 0.57?** The data is synthetic and churn depends on many weak signals. I checked that a much higher AUC
would come only from leakage: when I computed features after the churn dates, AUC rose to about 0.65, which is cheating. The honest
number is the one I report, with a 95% interval.

**How did you avoid data leakage?** All features are computed at a snapshot date (the day before the first churn) using only earlier data.
I proved it with a test that deletes all later data and rebuilds the features: the table did not change.

**Why no lifetime-value regression?** The payments table holds at most 15 payments per customer, and 83% of customers have no payments after
the training cutoff. A regression would learn that artefact. I used a transparent expected-value formula instead.

**How do you know the dashboard is right?** Totals match across three tools: SQL (8,000 customers, 24.89% churn), Python and Power BI.
I wrote the expected numbers down before building the pages and compared them.

**What would you do next?** Run a controlled retention test to measure the real save rate, add more months of churn data, and
use real campaign costs.

**What was the hardest part?** (Write your own answer: for many people it is matching each payment to the right subscription, or the leakage-safe snapshot.)

## Numbers to remember
- Customers {int(kpi['total_customers']):,}; active {int(kpi['active_customers']):,}; churn {kpi['churn_rate_pct']:.1f}%
- Monthly recurring revenue {kpi['monthly_recurring_revenue']:,.0f}; value at risk (12 months) {kpi['value_at_risk_12m']:,.0f}
- High-risk customers {int(kpi['high_risk_customers']):,}
"""
(DOCS / "presentation_notes.md").write_text(notes)
print("saved docs/presentation_notes.md")

saved docs/presentation_notes.md


In [12]:
# ---------- CELL 5: README.md ----------
readme = """# Netflix Customer Churn Intelligence Project

End-to-end analytics project: data cleaning (Python), database and 37 SQL business questions (MySQL),
exploratory analysis, churn model, segmentation, customer value, text themes, and a Power BI dashboard.

**Tools (all free):** Google Colab, MySQL Community Server + Workbench, Power BI Desktop, Google Drive for Desktop.
**Analysis date (as-of):** 2026-06-30. The data is synthetic.

## Folder structure
| Folder | Contents |
|---|---|
| `data/raw` | Original CSVs from the zip, never edited |
| `data/clean` | Cleaned CSVs (MySQL loads these) |
| `data/sql_exports` | Result CSVs exported from MySQL Workbench |
| `data/features` | Customer-level feature tables for the model |
| `data/model_outputs` | Predictions, segments, customer value, text themes |
| `data/bi_ready` | The only folder Power BI reads |
| `database` | Numbered `.sql` scripts |
| `notebooks` | Colab notebooks 01 to 15 |
| `src` | Helper code (`cleaning_helpers.py`, `features.py`) |
| `models` | Saved models |
| `reports` | Charts, evaluation report, final report, PDF |
| `docs` | Project log, audits, cleaning log, feature and BI dictionaries |
| `powerbi` | `.pbix`, theme, Power Query scripts, DAX measures |

## Run order (to rebuild everything from the zip file)
| Step | Tool | File | Produces |
|---|---|---|---|
| 1 | Colab | `notebooks/01_setup_and_audit` | Folders, raw data, audit |
| 2-3 | Colab | `02_cleaning_part1`, `03_cleaning_part2` | `data/clean/*.csv` |
| 4 | MySQL | `database/01..04_*.sql` (replace the folder placeholder once in `02_load_data.sql`) | Database `netflix_churn` |
| 5-7 | MySQL | `database/05..07_*.sql` | Answers to Q1-Q37 and exports in `data/sql_exports` |
| 8 | Colab | `08_eda` | Charts, `reports/eda_churn_by_group.csv` |
| 9 | Colab | `09_feature_engineering` | `data/features/*` |
| 10 | Colab | `10_churn_model` | Model, `churn_predictions.csv` |
| 11 | Colab | `11_segmentation` | `customer_segments.csv`, `segment_profile.csv` |
| 12 | Colab | `12_value_and_evaluation` | `customer_value.csv`, evaluation report |
| 13 | Colab | `13_text_themes_and_bi_tables` | Text themes, `data/bi_ready/*` |
| 14 | Colab + Power BI | `14_powerbi_setup_files`, then the `.pbix` | Power Query scripts, DAX, pages 1-3 |
| 15 | Colab + Power BI | `15_final_deliverables`, then the `.pbix` | Pages 4-7, PDF, report, README |

Every notebook starts with a small cell that finds `PROJECT_ROOT.marker` by walking up the folders, so the project works wherever it is copied.

## Refreshing the dashboard
Open `powerbi/netflix_churn_dashboard.pbix`, set the `DataFolder` parameter to your `data/bi_ready` folder
(Home > Transform data > Edit parameters), then Home > Refresh.

## Honest limits
Synthetic data; weak model signal (AUC about 0.57); churn dates only from 2025-10; payments capped at 15 per customer; templated text.
See `reports/Final_Project_Report.md`.
"""
(ROOT / "README.md").write_text(readme)
print("README.md written")

README.md written


In [18]:
# ---------- CELL 6: project manifest ----------
expected_files = (
    [f"data/raw/{n}.csv" for n in ["customers", "subscription_plans", "subscriptions", "content", "viewing_activity",
                                    "payments", "support_tickets", "customer_feedback", "churn_labels"]]
  + [f"data/clean/{n}.csv" for n in ["customers", "subscription_plans", "subscriptions", "content", "viewing_activity",
                                      "payments", "support_tickets", "customer_feedback", "churn_labels"]]
  + ["data/clean/as_of_date.txt"]
  + [f"database/{n}.sql" for n in ["01_create_tables", "02_load_data", "03_indexes_and_reference", "04_load_checks",
                                    "05_section_a_intermediate", "06_section_b_advanced", "07_section_c_expert"]]
  + [f"data/sql_exports/{n}.csv" for n in ["q04_churn_by_channel", "q06_top10_titles", "q20_cohort_retention",
        "q21_top5_churn_countries", "q30_country_churn_summary", "q31_at_risk_revenue_top50", "q33_winback_candidates",
        "q34_plan_migration", "q35_wau_mau", "q36_outreach_list", "q37_customer_health_score", "day04_row_counts"]]
  + ["src/cleaning_helpers.py", "src/features.py", "models/churn_model.joblib", "models/segment_model.joblib",
     "data/features/customer_features_train.csv", "data/features/customer_features_scoring.csv", "data/features/snapshot_dates.csv",
     "data/model_outputs/churn_predictions.csv", "data/model_outputs/customer_segments.csv", "data/model_outputs/segment_profile.csv",
     "data/model_outputs/customer_value.csv", "data/model_outputs/campaign_scenarios.csv",
     "data/model_outputs/text_themes_feedback.csv", "data/model_outputs/text_themes_tickets.csv",
     "reports/churn_model_metrics.csv", "reports/model_evaluation_report.md", "reports/Final_Project_Report.md",
     "reports/eda_churn_by_group.csv", "docs/project_log.md", "docs/cleaning_log.csv", "docs/feature_dictionary.csv",
     "docs/bi_data_dictionary.csv", "docs/bi_relationships.csv", "docs/eda_findings.md", "docs/presentation_notes.md",
     "powerbi/dax_measures.dax", "powerbi/netflix_theme.json", "README.md"]
  + [f"data/bi_ready/{n}.csv" for n in ["dim_customer", "dim_date", "dim_plan", "dim_content", "fact_payments", "fact_viewing",
        "fact_tickets", "fact_feedback", "fact_subscriptions", "kpi_summary", "model_segment_profile", "model_lift_by_decile",
        "model_campaign_scenarios", "model_text_theme_summary", "sql_cohort_retention", "sql_country_churn_summary",
        "sql_wau_mau", "sql_outreach_list", "sql_winback_candidates", "sql_plan_migration"]]
)
made_by_hand = ["powerbi/netflix_churn_dashboard.pbix", "reports/netflix_churn_dashboard.pdf"]     # created in Power BI

rows = []
for rel in expected_files + made_by_hand:
    path = ROOT / rel
    exists = path.exists()
    line_count = None
    if exists and path.suffix == ".csv":
        with open(path, "rb") as fh:
            line_count = max(sum(1 for _ in fh) - 1, 0)
    rows.append({"file": rel, "exists": exists, "size_kb": round(path.stat().st_size / 1024, 1) if exists else None,
                 "data_rows": line_count, "made_in": "Power BI" if rel in made_by_hand else "Python/SQL"})
manifest = pd.DataFrame(rows)
manifest.to_csv(DOCS / "project_manifest.csv", index=False)

missing = manifest[~manifest["exists"]]
print(f"{manifest['exists'].sum()} of {len(manifest)} expected files exist.")
if len(missing):
    print("\nMISSING:")
    print(missing[["file", "made_in"]].to_string(index=False))
else:
    print("Nothing is missing.")

88 of 88 expected files exist.
Nothing is missing.
